In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.metrics import (
    roc_auc_score, confusion_matrix, classification_report,
    precision_recall_curve
)

RANDOM_STATE = 42
TARGET = "delayed"  # change if needed

In [2]:
from catboost import CatBoostClassifier

def catboost_grid_search(X, y, cat_features, param_grid, n_splits=3):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)
    best_auc = -1
    best_params = None

    for params in param_grid:
        fold_aucs = []
        for tr_idx, val_idx in skf.split(X, y):
            X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
            y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]

            model = CatBoostClassifier(
                iterations=3000,                # large
                loss_function="Logloss",
                eval_metric="AUC",
                random_seed=42,
                verbose=False
            )

            model.fit(
                X_tr, y_tr,
                cat_features=cat_features,
                eval_set=(X_val, y_val),
                early_stopping_rounds=200,
                use_best_model=True
            )

            p = model.predict_proba(X_val)[:, 1]
            fold_aucs.append(roc_auc_score(y_val, p))

        mean_auc = float(np.mean(fold_aucs))
        if mean_auc > best_auc:
            best_auc = mean_auc
            best_params = params

    return best_params, best_auc

In [3]:
results = []

def add_row(name, y_test, y_prob, thr):
    y_pred = (y_prob >= thr).astype(int)
    rep = classification_report(y_test, y_pred, output_dict=True)
    results.append({
        "Model": name,
        "Threshold": thr,
        "ROC_AUC": roc_auc_score(y_test, y_prob),
        "Recall_Delayed": rep["1"]["recall"],
        "Precision_Delayed": rep["1"]["precision"],
        "F1_Delayed": rep["1"]["f1-score"]
    })

# After each evaluation call add_row(...)

In [4]:
def best_threshold_f1(y_true, y_prob):
    precision, recall, thresholds = precision_recall_curve(y_true, y_prob)
    f1 = 2 * (precision * recall) / (precision + recall + 1e-9)
    # thresholds is 1 shorter than precision/recall
    best_idx = np.nanargmax(f1[:-1])
    thr = float(thresholds[best_idx])
    return thr, float(f1[best_idx]), float(precision[best_idx]), float(recall[best_idx])

In [5]:
def eval_model(name, y_test, y_prob, thr=0.5):
    y_pred = (y_prob >= thr).astype(int)
    print(f"\n=== {name} ===")
    print("Threshold:", thr)
    print("ROC-AUC:", roc_auc_score(y_test, y_prob))
    print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
    print(classification_report(y_test, y_pred))
    print("Predicted delayed rate:", y_pred.mean(), "| Actual delayed rate:", y_test.mean())

In [5]:
df_tree = pd.read_csv("model_ready_tree.csv")
X = df_tree.drop(columns=[TARGET])
y = df_tree[TARGET].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train, test_size=0.2, stratify=y_train, random_state=RANDOM_STATE
)

In [6]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1)

rf_grid = {
    "n_estimators": [400, 700],
    "max_depth": [10, 14, 18],
    "min_samples_split": [10, 20],
    "min_samples_leaf": [10, 20],
    "max_features": ["sqrt", 0.5],
    "class_weight": ["balanced_subsample"]
}

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)

rf_gs = GridSearchCV(
    rf, rf_grid, scoring="average_precision",  # PR-AUC aligns better with delayed class
    cv=cv, n_jobs=-1, verbose=1, refit=True
)

rf_gs.fit(X_tr, y_tr)
rf_best = rf_gs.best_estimator_
print("RF best params:", rf_gs.best_params_)

Fitting 3 folds for each of 48 candidates, totalling 144 fits
RF best params: {'class_weight': 'balanced_subsample', 'max_depth': 18, 'max_features': 0.5, 'min_samples_leaf': 10, 'min_samples_split': 10, 'n_estimators': 700}


In [7]:
rf_val_prob = rf_best.predict_proba(X_val)[:, 1]
rf_thr, rf_f1, rf_p, rf_r = best_threshold_f1(y_val, rf_val_prob)
print("RF best thr:", rf_thr, "Val F1:", rf_f1, "Val P:", rf_p, "Val R:", rf_r)

rf_test_prob = rf_best.predict_proba(X_test)[:, 1]
eval_model("Random Forest (tuned)", y_test, rf_test_prob, rf_thr)

add_row("Random Forest (tuned)", y_test, rf_test_prob, rf_thr)

RF best thr: 0.5836484230005784 Val F1: 0.5234212018594436 Val P: 0.4651685393258427 Val R: 0.5983523630582455

=== Random Forest (tuned) ===
Threshold: 0.5836484230005784
ROC-AUC: 0.7960312756916639
Confusion Matrix:
 [[28133  5981]
 [ 3521  5127]]
              precision    recall  f1-score   support

           0       0.89      0.82      0.86     34114
           1       0.46      0.59      0.52      8648

    accuracy                           0.78     42762
   macro avg       0.68      0.71      0.69     42762
weighted avg       0.80      0.78      0.79     42762

Predicted delayed rate: 0.25976334128431783 | Actual delayed rate: 0.2022356297647444


In [8]:
# If not already defined from RF section:
df_tree = pd.read_csv("model_ready_tree.csv")
X = df_tree.drop(columns=[TARGET])
y = df_tree[TARGET].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train, test_size=0.2, stratify=y_train, random_state=RANDOM_STATE
)

neg, pos = (y_tr == 0).sum(), (y_tr == 1).sum()
scale_pos_weight = neg / max(pos, 1)

In [9]:
from xgboost import XGBClassifier

xgb = XGBClassifier(
    random_state=RANDOM_STATE,
    n_jobs=-1,
    tree_method="hist",
    eval_metric="auc",
    scale_pos_weight=scale_pos_weight
)

xgb_grid = {
    "n_estimators": [600, 1000],
    "max_depth": [4, 6],
    "learning_rate": [0.03, 0.05],
    "subsample": [0.8, 1.0],
    "colsample_bytree": [0.8, 1.0],
    "min_child_weight": [1, 5],
    "reg_lambda": [1, 3],
}

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)

xgb_gs = GridSearchCV(
    xgb, xgb_grid,
    scoring="average_precision",
    cv=cv, n_jobs=-1, verbose=1, refit=True
)

xgb_gs.fit(X_tr, y_tr)
xgb_best = xgb_gs.best_estimator_
print("XGB best params:", xgb_gs.best_params_)

Fitting 3 folds for each of 128 candidates, totalling 384 fits
XGB best params: {'colsample_bytree': 1.0, 'learning_rate': 0.05, 'max_depth': 6, 'min_child_weight': 1, 'n_estimators': 1000, 'reg_lambda': 3, 'subsample': 0.8}


In [10]:
xgb_val_prob = xgb_best.predict_proba(X_val)[:, 1]
xgb_thr, xgb_f1, xgb_p, xgb_r = best_threshold_f1(y_val, xgb_val_prob)
print("XGB best thr:", xgb_thr, "Val F1:", xgb_f1, "Val P:", xgb_p, "Val R:", xgb_r)

xgb_test_prob = xgb_best.predict_proba(X_test)[:, 1]
eval_model("XGBoost (tuned)", y_test, xgb_test_prob, xgb_thr)

add_row("XGBoost (tuned)", y_test, xgb_test_prob, xgb_thr)

XGB best thr: 0.5903108716011047 Val F1: 0.5369760938905342 Val P: 0.4707163074243414 Val R: 0.6249458014163897

=== XGBoost (tuned) ===
Threshold: 0.5903108716011047
ROC-AUC: 0.8053997691366982
Confusion Matrix:
 [[27918  6196]
 [ 3324  5324]]
              precision    recall  f1-score   support

           0       0.89      0.82      0.85     34114
           1       0.46      0.62      0.53      8648

    accuracy                           0.78     42762
   macro avg       0.68      0.72      0.69     42762
weighted avg       0.81      0.78      0.79     42762

Predicted delayed rate: 0.26939806370141717 | Actual delayed rate: 0.2022356297647444


In [10]:
df_cb = pd.read_csv("model_ready_catboost.csv")
X = df_cb.drop(columns=[TARGET])
y = df_cb[TARGET].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train, test_size=0.2, stratify=y_train, random_state=RANDOM_STATE
)

cat_cols = X_tr.select_dtypes(include=["object"]).columns.tolist()
cat_features = [X_tr.columns.get_loc(c) for c in cat_cols]

neg, pos = (y_tr == 0).sum(), (y_tr == 1).sum()
w1 = neg / max(pos, 1)

In [7]:
df_nn = pd.read_csv("model_ready_nn.csv")
X = df_nn.drop(columns=[TARGET])
y = df_nn[TARGET].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train, test_size=0.2, stratify=y_train, random_state=RANDOM_STATE
)

In [8]:
from sklearn.neural_network import MLPClassifier

mlp = MLPClassifier(
    random_state=RANDOM_STATE,
    early_stopping=True,
    validation_fraction=0.15,
    max_iter=80
)

mlp_grid = {
    "hidden_layer_sizes": [(128,64), (256,128,64)],
    "alpha": [1e-4, 5e-4, 1e-3],
    "learning_rate_init": [1e-3, 5e-4],
    "batch_size": [256, 512],
}

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)

mlp_gs = GridSearchCV(
    mlp, mlp_grid,
    scoring="average_precision",
    cv=cv, n_jobs=-1, verbose=1, refit=True
)

mlp_gs.fit(X_tr, y_tr)
mlp_best = mlp_gs.best_estimator_
print("MLP best params:", mlp_gs.best_params_)

Fitting 3 folds for each of 24 candidates, totalling 72 fits
MLP best params: {'alpha': 0.001, 'batch_size': 256, 'hidden_layer_sizes': (256, 128, 64), 'learning_rate_init': 0.0005}


In [9]:
mlp_val_prob = mlp_best.predict_proba(X_val)[:, 1]
mlp_thr, mlp_f1, mlp_p, mlp_r = best_threshold_f1(y_val, mlp_val_prob)
print("MLP best thr:", mlp_thr, "Val F1:", mlp_f1, "Val P:", mlp_p, "Val R:", mlp_r)

mlp_test_prob = mlp_best.predict_proba(X_test)[:, 1]
eval_model("MLP (tuned)", y_test, mlp_test_prob, mlp_thr)

add_row("MLP (tuned)", y_test, mlp_test_prob, mlp_thr)

MLP best thr: 0.21849377841468492 Val F1: 0.5252620974846353 Val P: 0.44500200722601363 Val R: 0.6408440526087585

=== MLP (tuned) ===
Threshold: 0.21849377841468492
ROC-AUC: 0.7933491364889242
Confusion Matrix:
 [[27006  7108]
 [ 3132  5516]]
              precision    recall  f1-score   support

           0       0.90      0.79      0.84     34114
           1       0.44      0.64      0.52      8648

    accuracy                           0.76     42762
   macro avg       0.67      0.71      0.68     42762
weighted avg       0.80      0.76      0.78     42762

Predicted delayed rate: 0.29521537813946963 | Actual delayed rate: 0.2022356297647444
